In [1]:
!hostname

cn063.delta.ncsa.illinois.edu


In [2]:
import scanpy as sc
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import os
os.chdir('/projects/bgdb/asachan/methods/maxToki-multimodal')  # directory containing utils.py
import sys
import logging
import warnings

export_dir = "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human"
raw_dir = "/work/hdd/bgdb/asachan/datasets_hdd/SKM_multimodal_ageing/raw_files"
out_tmp = '/projects/bgdb/asachan/tmp'

In [4]:
pd.set_option('mode.string_storage', 'python')

# RNA preproc

In [ ]:
adata_rna = f'{export_dir}/rna_objects/rna_female_type2_counts_added.h5ad'

In [ ]:
rna_adata = sc.read_h5ad(adata_rna)

#### Downsampling (not random, based on dna-damage and metabolic scoring trends, so that we can focus on signals of interest)

In [ ]:
# filter out OM2 cells
rna_adata = rna_adata[rna_adata.obs['sample'] != 'OM2'].copy()
# filter out YM2_1 cells
rna_adata = rna_adata[rna_adata.obs['orig.ident']s != 'YM2_1'].copy()

In [ ]:
#drop OM9_D* cells from rna_adata (these are Gluteus Medius cells)
rna_adata = rna_adata[~rna_adata.obs['orig.ident'].str.contains('OM9_D')]

In [ ]:
rna_adata.obs['orig.ident'].value_counts()

In [ ]:
rna_adata.obs['sample'].value_counts()

### adding raw counts layer for these retained cells

In [ ]:
rna_adata

In [ ]:
rna_adata.X.max() # float means the counts are normalized

#### Strip suffix per replicate library and intersect

In [ ]:
import gzip, glob, os, re
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp
import anndata as ad

raw_root = f"{raw_dir}/female_type2_maxtoki"

orig_to_lib = {
    'OM6_1':  'OM6_VL_snRNA_seq_1',
    'OM6_2':  'OM6_VL_snRNA_seq_2',
    'OM6_3':  'OM6_VL_snRNA_seq_3',
    'OM6_4':  'OM6_VL_snRNA_seq_4',
    'OM9_N1': 'OM9_VL_snRNA_seq_1',
    'OM9_N2': 'OM9_VL_snRNA_seq_2',
    'OM9_N3': 'OM9_VL_snRNA_seq_3',
    'OM9_N4': 'OM9_VL_snRNA_seq_4',
    'YM2_2':  'YM2_ST_snRNA_seq_2',
    'YM2_3':  'YM2_ST_snRNA_seq_3',
    'P26_1':  'P26_ST_snRNA_seq_1',
    'P26_2':  'P26_ST_snRNA_seq_2',
    'P26_4':  'P26_ST_snRNA_seq_3',
    'P26_5':  'P26_ST_snRNA_seq_4',
}
assert set(orig_to_lib) == set(rna_adata.obs['orig.ident'].unique())

#### convert adata cells to raw library barcodes for intersection

pat = re.compile(r'^(CELL\d+_N\d+)(_.+)$')

# 1. Find merge suffix per orig.ident
suffix_per_orig = {}
for oid in rna_adata.obs['orig.ident'].unique():
    cells = rna_adata.obs_names[rna_adata.obs['orig.ident'] == oid]
    suffixes = {pat.match(c).group(2) for c in cells}
    assert len(suffixes) == 1, f"{oid}: multiple suffixes {suffixes}"
    suffix_per_orig[oid] = suffixes.pop()

# 2. Strip the suffix → set of raw-style barcodes per orig.ident
oid_to_stripped = {}
for oid, suf in suffix_per_orig.items():
    cells = rna_adata.obs_names[rna_adata.obs['orig.ident'] == oid]
    oid_to_stripped[oid] = {c[:-len(suf)] for c in cells}

# --- helpers ---
def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert len(hits) >= 1, f"no file matching {pattern}"
    return hits[0]

def load_raw_barcodes(path):
    return pd.read_csv(path, header=None, sep='\t')[0].tolist()

# --- per orig.ident loader ---
def load_library(oid, lib_prefix, merge_suffix):
    bc_path   = find_one(f"{raw_root}/**/{lib_prefix}_barcodes.tsv*")
    feat_path = find_one(f"{raw_root}/**/{lib_prefix}_features.tsv*")
    mat_path  = find_one(f"{raw_root}/**/{lib_prefix}_matrix.mtx*")

    raw_bcs = load_raw_barcodes(bc_path)

    # subset assertion
    expected = oid_to_stripped[oid]
    missing  = expected - set(raw_bcs)
    assert not missing, (
        f"{oid}: {len(missing)}/{len(expected)} adata cells absent from "
        f"{os.path.basename(bc_path)}"
    )

    feats    = pd.read_csv(feat_path, header=None, sep='\t')
    gene_col = 1 if feats.shape[1] >= 2 else 0
    genes    = feats[gene_col].astype(str).tolist()

    X = sio.mmread(mat_path).T.tocsr()                   # cells × genes
    assert X.shape == (len(raw_bcs), len(genes))

    # rename raw barcodes back to adata's merge-suffixed form
    new_names = [bc + merge_suffix for bc in raw_bcs]

    a = ad.AnnData(
        X=X,
        obs=pd.DataFrame({'orig.ident': oid}, index=pd.Index(new_names)),
        var=pd.DataFrame(index=pd.Index(genes)),
    )
    a.var_names_make_unique()

    # subset adata cells from this library
    keep = rna_adata.obs_names.intersection(a.obs_names)
    assert len(keep) == (rna_adata.obs['orig.ident'] == oid).sum(), \
        f"{oid}: expected {(rna_adata.obs['orig.ident']==oid).sum()} cells, got {len(keep)}"
    a = a[keep].copy()
    return a

In [ ]:
# Quick check of barcodes in adata being a complete subset of barcodes in raw library files (P26, YM2, OM6, OM9)
sample_name = 'OM9'
for oid in [f'{sample_name}_N1', f'{sample_name}_N2', f'{sample_name}_N3', f'{sample_name}_N4']:
    print(oid, len(oid_to_stripped[oid]),
          'cells:', list(oid_to_stripped[oid])[:3])

sample_raw_dir = f"{raw_root}/{sample_name}"
sample_libs = [f'{sample_name}_VL_snRNA_seq_1', f'{sample_name}_VL_snRNA_seq_2',
            f'{sample_name}_VL_snRNA_seq_3', f'{sample_name}_VL_snRNA_seq_4']
sample_oids = [f'{sample_name}_N1', f'{sample_name}_N2', f'{sample_name}_N3', f'{sample_name}_N4']

# Load raw barcode sets
raw_sample = {}
for lib in sample_libs:
    raw_sample[lib] = set(load_raw_barcodes(f"{sample_raw_dir}/{lib}_barcodes.tsv.gz"))
    print(f"{lib}: {len(raw_sample[lib])} barcodes")

# 4 x 4 subset table: rows = orig.ident, cols = raw library
rows = []
for oid in sample_oids:
    expected = oid_to_stripped[oid]   # adata cells (suffix-stripped) for this orig.ident
    row = {'orig.ident': oid, 'n_adata': len(expected)}
    for lib in sample_libs:
        n = len(expected & raw_sample[lib])
        row[lib] = f"{n}/{len(expected)}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# --- load all libraries ---
per_lib = {}
for oid, lib in orig_to_lib.items():
    a = load_library(oid, lib, suffix_per_orig[oid])
    per_lib[oid] = a
    print(f"{oid:7s} ← {lib:25s}  {a.shape}")

In [ ]:
# --- concat per-library AnnDatas ---
raw_all = ad.concat(
    list(per_lib.values()),
    join='outer',           # union of genes across libraries
    label='lib_src',
    index_unique=None,      # cells already globally unique via merge suffix
    merge='same',
)
print(f"concat: {raw_all.shape}")

# --- align to adata cell order ---
assert set(raw_all.obs_names) == set(rna_adata.obs_names)
raw_all = raw_all[rna_adata.obs_names].copy()

In [ ]:
# --- align gene axis to adata.var_names ---
gene_loc = pd.Index(raw_all.var_names).get_indexer(rna_adata.var_names)
present  = gene_loc >= 0
n_missing = (~present).sum()
print(f"genes shared: {present.sum()}/{rna_adata.n_vars}  (missing: {n_missing})")

src      = raw_all.X.tocsc()[:, gene_loc[present]]
counts   = sp.lil_matrix((rna_adata.n_obs, rna_adata.n_vars), dtype=src.dtype)
counts[:, np.where(present)[0]] = src
counts   = counts.tocsr()

rna_adata.layers['counts'] = counts

In [ ]:
# --- verifications ---
totals = np.asarray(counts.sum(axis=1)).ravel()
nfeat  = np.asarray((counts > 0).sum(axis=1)).ravel()
sub    = counts[:1000].toarray()

print(f"integer?                        {(sub == sub.astype(int)).all()}")
print(f"max value:                      {sub.max()}")
print(f"corr(layer.sum, nCount_RNA)   = {np.corrcoef(totals, rna_adata.obs['nCount_RNA'])[0,1]:.6f}")
print(f"corr(nFeat_layer, nFeature_RNA) = {np.corrcoef(nfeat,  rna_adata.obs['nFeature_RNA'])[0,1]:.6f}")
print(f"max |Δ nCount|:  {np.abs(totals - rna_adata.obs['nCount_RNA'].values).max()}")
print(f"max |Δ nFeature|:{np.abs(nfeat  - rna_adata.obs['nFeature_RNA'].values).max()}")

### Inspecting the prepared adata rna object

In [ ]:
rna_adata

In [ ]:
rna_adata.X = rna_adata.layers['counts']
display(rna_adata.X.max())

In [ ]:
import numpy as np
import scipy.sparse as sp

# Preserve original
rna_adata.layers['counts_float'] = rna_adata.layers['counts']

# Rounded integer version
counts = rna_adata.layers['counts']
if sp.issparse(counts):
    counts_int = counts.copy()
    counts_int.data = np.round(counts_int.data).astype(np.int32)
    counts_int.eliminate_zeros()
    counts_int = counts_int.tocsr()
else:
    counts_int = np.round(counts).astype(np.int32)
    counts_int = sp.csr_matrix(counts_int)

rna_adata.layers['counts'] = counts_int

# Verify
print(f"dtype: {counts_int.dtype}")
print(f"sparse: {sp.issparse(counts_int)}, format: {counts_int.format}")
print(f"max: {counts_int.max()}, min: {counts_int.data.min() if counts_int.nnz else 0}")
print(f"nnz: {counts_int.nnz} (was {counts.nnz})")
print(f"memory: {counts_int.data.nbytes / 1e6:.1f} MB")

In [ ]:
import pandas as pd
import numpy as np
import anndata

# 1. Disable Arrow-backed strings globally for this session
pd.set_option('future.infer_string', False)
pd.set_option('mode.string_storage', 'python')

# 2. Diagnose
print(f"anndata version:    {anndata.__version__}")
print(f"future.infer_string: {pd.get_option('future.infer_string')}")
print(f"mode.string_storage: {pd.get_option('mode.string_storage')}")

def nuke_arrow_full(df):
    df = df.copy()
    # Index
    df.index = pd.Index(pd.array(df.index.tolist(), dtype=object),
                        name=df.index.name)
    # Columns
    df.columns = pd.Index(pd.array(df.columns.tolist(), dtype=object))
    # Each column
    for col in list(df.columns):
        s = df[col]
        if isinstance(s.dtype, pd.CategoricalDtype):
            # Rebuild categorical with object-dtype categories
            cats = pd.array(s.cat.categories.tolist(), dtype=object)
            new_cats = pd.Index(cats)
            df[col] = pd.Categorical(
                s.astype(object),                # round-trip to plain strings
                categories=new_cats,
                ordered=s.cat.ordered,
            )
        elif 'arrow' in type(s.array).__name__.lower() or 'string' in str(s.dtype).lower():
            df[col] = pd.array(s.tolist(), dtype=object)
    return df

rna_adata.obs = nuke_arrow_full(rna_adata.obs)
rna_adata.var = nuke_arrow_full(rna_adata.var)

# Verify all categoricals
for col in rna_adata.obs.columns:
    s = rna_adata.obs[col]
    if isinstance(s.dtype, pd.CategoricalDtype):
        backing = type(s.cat.categories.array).__name__
        print(f"obs.{col}: categories backing = {backing}")

# Same .uns gotcha — color arrays etc. live there and can also be ArrowStringArray
# Just drop the entire `_colors` set; scanpy regenerates them on demand
for k in list(rna_adata.uns.keys()):
    if k.endswith('_colors'):
        del rna_adata.uns[k]

out_path = f"{export_dir}/rna_female_type2_counts_added.h5ad"
rna_adata.write_h5ad(out_path)
print(f"saved: {out_path}")

#### Viz

In [ ]:
rna_adata

In [ ]:
sc.pl.embedding(rna_adata, basis='X_umap', color=['Annotation', 'Pseudotime_typeII', 'age'])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scipy.stats as st
import seaborn as sns

def _get_counts(adata, layer=None):
    X = adata.layers[layer] if layer else adata.X
    return X if sp.issparse(X) else sp.csr_matrix(X)

def _mean_var(X):
    # Sparse-safe per-feature mean and variance over cells
    n = X.shape[0]
    mean = np.asarray(X.mean(axis=0)).ravel()
    # E[X^2] - E[X]^2
    sqmean = np.asarray(X.multiply(X).mean(axis=0)).ravel()
    var = sqmean - mean**2
    return mean, var
    
def rna_diagnostics(adata_rna, layer='counts', n_genes_hist=8, save=None):
    X = _get_counts(adata_rna, layer=layer)
    mean, var = _mean_var(X)
    keep = mean > 1e-4                    # drop ~all-zero genes
    mean, var = mean[keep], var[keep]
    gene_names = adata_rna.var_names[keep]

    # NB MoM dispersion: var = mu + alpha * mu^2  =>  alpha_hat = (var - mu)/mu^2
    alpha = np.maximum((var - mean) / mean**2, 1e-8)

    # Observed zero fraction per gene
    n_cells = X.shape[0]
    zero_frac_obs = 1 - np.asarray((X != 0).sum(axis=0)).ravel()[keep] / n_cells
    # NB-predicted zero fraction with alpha=1/r (shape r): P(X=0) = (r/(r+mu))^r
    r = 1 / alpha
    zero_frac_nb = (r / (r + mean)) ** r

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))

    # (1) Mean-variance on log-log; reference Poisson (var=mean) and NB (var=mean+alpha*mean^2)
    ax = axes[0, 0]
    ax.scatter(mean, var, s=4, alpha=0.3, rasterized=True)
    grid = np.logspace(np.log10(mean.min()), np.log10(mean.max()), 200)
    ax.plot(grid, grid, 'k--', label='Poisson (var = μ)')
    ax.plot(grid, grid + np.median(alpha) * grid**2, 'r-',
            label=f'NB (median α = {np.median(alpha):.2f})')
    ax.set(xscale='log', yscale='log', xlabel='mean (per gene)', ylabel='variance')
    ax.legend(); ax.set_title('RNA mean–variance (overdispersion)')

    # (2) Dispersion histogram
    ax = axes[0, 1]
    ax.hist(np.log10(alpha), bins=80)
    ax.set(xlabel='log10(α̂) per gene', ylabel='# genes',
           title='Per-gene dispersion (α≫0)')

    # (3) Observed vs NB-predicted zero fraction — points above y=x ⇒ zero-inflation
    ax = axes[1, 0]
    ax.scatter(zero_frac_nb, zero_frac_obs, s=4, alpha=0.3, rasterized=True)
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set(xlabel='NB-predicted P(X=0)', ylabel='observed zero fraction',
           title='Zero-inflation check')

    # (4) Per-gene count histograms for a few representative genes (varied mean)
    ax = axes[1, 1]
    idx = np.argsort(mean)[np.linspace(0, len(mean) - 1, n_genes_hist).astype(int)]
    for i in idx:
        counts = np.asarray(X[:, np.where(keep)[0][i]].todense()).ravel()
        ax.hist(counts, bins=np.arange(0, counts.max() + 2) - 0.5,
                histtype='step', density=True,
                label=f'{gene_names[i]} (μ={mean[i]:.2f})')
    ax.set(xlabel='count', ylabel='density', yscale='log',
           title='Per-gene count distributions (sample of genes)')
    ax.legend(fontsize=7)

    fig.tight_layout()
    if save: fig.savefig(save, dpi=200, bbox_inches='tight')
    return dict(mean=mean, var=var, alpha=alpha,
                zero_frac_obs=zero_frac_obs, zero_frac_nb=zero_frac_nb)

In [ ]:
rna_stats  = rna_diagnostics(rna_adata,  layer='counts', save='/projects/bgdb/asachan/methods/maxToki-multimodal/figs/rna_diagnostics.png')
print(f"Median RNA dispersion α: {np.median(rna_stats['alpha']):.3f}")
print(f"Genes with obs zero frac > NB-predicted by >0.1: "
      f"{((rna_stats['zero_frac_obs'] - rna_stats['zero_frac_nb']) > 0.1).sum()}")

# ATAC preproc

In [ ]:
import snapatac2 as snap

In [ ]:
atac_peaks_adata = '/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/atac_objects/atac_fiber/atac_peaks.h5ad'
out_tmp = '/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/tmp'

In [ ]:
atac_peaks_adata = sc.read_h5ad(atac_peaks_adata)
#rename the obs 'Sample' to 'replicates' 
atac_peaks_adata.obs.rename(columns={'Sample': 'replicates'}, inplace=True)

In [ ]:
atac_peaks_adata

In [ ]:
atac_peaks_adata.X.max()

#### Downsample to match rna samples and replicates

In [ ]:
# retain only samples of interest 
atac_peaks_adata = atac_peaks_adata[atac_peaks_adata.obs['sample'].isin(['OM6', 'OM9', 'YM2', 'P26'])].copy()


In [ ]:
exclude = ['YM2_semitendinosus_1', 'OM9_D1', 'OM9_D2', 'OM9_D3', 'OM9_D4']
atac_peaks_adata = atac_peaks_adata[~atac_peaks_adata.obs['orig.ident'].isin(exclude)].copy()

In [ ]:
atac_peaks_adata.obs['sample'].value_counts()

In [ ]:
# filter cells to retain only type 2 cells - has II in its atring for annotation
atac_peaks_adata = atac_peaks_adata[atac_peaks_adata.obs['Annotation'].str.contains('II')].copy()
sc.pl.embedding(atac_peaks_adata, basis='X_UMAP', color=['Annotation'])


### check counts in the peak matrix (distribution)

In [ ]:
atac_peaks_adata = sc.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/atac_objects/atac_fiber/atac_female_type2.h5ad')

In [ ]:
atac_peaks_adata

In [ ]:
atac_peaks_adata.obs['sample'].value_counts()

In [ ]:
atac_peaks_adata.obs['orig.ident'].value_counts()

In [ ]:
atac_peaks_adata.obs['replicates'].value_counts()

In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

X = atac_peaks_adata.X
if not sp.issparse(X): X = sp.csr_matrix(X)

# --- numeric summary ---
nz = X.data
n_cells, n_peaks = X.shape
frac_gt1 = (nz > 1).mean()
val_dist = np.bincount(nz.astype(int))[:11]   # counts of 0..10 (index 0 ignored)

print(f"shape:                 {n_cells} cells × {n_peaks} peaks")
print(f"nonzero entries:       {nz.size:,}  (sparsity = {1 - nz.size/(n_cells*n_peaks):.4f})")
print(f"max value:             {nz.max()}")
print(f"% of nonzero == 1:     {(nz == 1).mean():.2%}")
print(f"% of nonzero  > 1:     {frac_gt1:.2%}    ← binarization-loss")
print(f"value distribution (1..10):")
for v in range(1, 11):
    n = (nz == v).sum()
    if n: print(f"  ={v:2d}: {n:>10,}  ({n/nz.size:.2%})")

# --- per-peak mean & variance, sparse-safe ---
mean = np.asarray(X.mean(axis=0)).ravel()
sqmean = np.asarray(X.multiply(X).mean(axis=0)).ravel()
var  = sqmean - mean**2
keep = mean > 1e-4

# --- 3-panel diagnostic ---
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

# (1) nonzero value distribution — the headline plot
counts_by_v = [(nz == v).sum() for v in range(1, 11)]
ax[0].bar(range(1, 11), np.array(counts_by_v) / nz.size, color='steelblue')
ax[0].set(xlabel='count value (nonzero only)', ylabel='fraction of nonzero entries',
          title=f'Value distribution\n{frac_gt1:.1%} of nonzero entries are >1')
ax[0].set_xticks(range(1, 11))

# (2) mean–variance vs Bernoulli reference
ax[1].scatter(mean[keep], var[keep], s=3, alpha=0.25, rasterized=True, label='peaks')
grid = np.linspace(0, mean[keep].max(), 200)
ax[1].plot(grid, grid * (1 - grid), 'r-', lw=2, label='Bernoulli: p(1−p)')
ax[1].plot(grid, grid, 'k--', lw=1, label='Poisson: var = μ')
ax[1].set(xlabel='per-peak mean (over cells)', ylabel='variance',
          title='Mean–variance (Bernoulli ⇒ on red curve)')
ax[1].legend()

# (3) per-peak "open fraction" — empirical p̂ if data were binary
p_peak = np.asarray((X > 0).sum(axis=0)).ravel() / n_cells
ax[2].hist(p_peak, bins=80, color='steelblue')
ax[2].set(xlabel='per-peak open fraction p̂ = P(count > 0)',
          ylabel='# peaks', title='Empirical p per peak')

fig.tight_layout()
plt.show()

In [ ]:
atac_peaks_adata.layers['Tn5_insertion_counts'] = atac_peaks_adata.X.copy()

In [ ]:
atac_peaks_adata

In [ ]:
import pandas as pd
import numpy as np

# Make sure these options are set in the session
pd.set_option('future.infer_string', False)
pd.set_option('mode.string_storage', 'python')

def nuke_arrow_full(df):
    """Strip ArrowStringArray from df's index, columns, and string-like cols."""
    df = df.copy()
    df.index = pd.Index(pd.array(df.index.tolist(), dtype=object),
                        name=df.index.name)
    df.columns = pd.Index(pd.array(df.columns.tolist(), dtype=object))
    for col in list(df.columns):
        s = df[col]
        if isinstance(s.dtype, pd.CategoricalDtype):
            cats = pd.array(s.cat.categories.tolist(), dtype=object)
            df[col] = pd.Categorical(
                s.astype(object),
                categories=pd.Index(cats),
                ordered=s.cat.ordered,
            )
        elif 'arrow' in type(s.array).__name__.lower() or 'string' in str(s.dtype).lower():
            df[col] = pd.array(s.tolist(), dtype=object)
    return df

def safe_write_h5ad(adata, path, drop_raw=True, drop_colors=True):
    """Coerce ArrowStringArray everywhere, then write."""
    adata.obs = nuke_arrow_full(adata.obs)
    adata.var = nuke_arrow_full(adata.var)
    if drop_raw and adata.raw is not None:
        del adata.raw
    if drop_colors:
        for k in list(adata.uns.keys()):
            if k.endswith('_colors'):
                del adata.uns[k]
    adata.write_h5ad(path)
    print(f"saved: {path}")

safe_write_h5ad(
    atac_peaks_adata,
    "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/atac_objects/atac_fiber/atac_female_type2.h5ad",
)

# Construct combined mudata

In [ ]:
rna_adata_filtered = sc.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/rna_objects/rna_female_type2_counts_added.h5ad')
atac_adata_filtered = sc.read_h5ad('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/atac_objects/atac_fiber/atac_female_type2.h5ad')

#### filter rna features

In [ ]:
# RNA: 4000 HVGs on raw counts
sc.pp.highly_variable_genes(
    rna_adata_filtered, n_top_genes=4000, layer='counts', flavor='seurat_v3'
)
rna_adata_filtered = rna_adata_filtered[:, rna_adata_filtered.var['highly_variable']].copy()

In [ ]:
rna_adata_filtered

In [ ]:
# plot dotplot of pdk4 expression across sample cells
sc.pl.dotplot(rna_adata_filtered, 'PDK4', groupby='sample')

In [ ]:
rna_adata_filtered.obs['orig.ident'].value_counts()

In [ ]:
#rename P26_4 to P26_3 and P26_5 to P26_4 in orig.ident column of rna_adata_filtered
rna_adata_filtered.obs['orig.ident'] = rna_adata_filtered.obs['orig.ident'].str.replace('P26_4', 'P26_3')
rna_adata_filtered.obs['orig.ident'] = rna_adata_filtered.obs['orig.ident'].str.replace('P26_5', 'P26_4')
rna_adata_filtered.obs['orig.ident'].value_counts()

#### filter atac features

In [ ]:
# ATAC: peaks open in ≥1% of cells (a standard threshold)
min_cells = max(int(0.01 * atac_adata_filtered.n_obs), 5)   # ~78 cells here
sc.pp.filter_genes(atac_adata_filtered, min_cells=min_cells)
print(f"ATAC peaks after filter: {atac_adata_filtered.n_vars}")  # expect ~80–150k

In [ ]:
import numpy as np
n_open = np.asarray((atac_adata_filtered.X > 0).sum(axis=0)).ravel()
print(f"Peaks open in 1–10 cells:    {((n_open >= 1) & (n_open <= 10)).sum()}")  
print(f"Peaks open in 11–78 cells:   {((n_open > 10) & (n_open < 79)).sum()}")
print(f"Peaks open in ≥78 cells (kept): {(n_open >= 78).sum()}")
print(f"Median open fraction (kept):   {np.median(n_open[n_open >= 78]) / atac_adata_filtered.n_obs:.3%}")

In [ ]:
atac_adata_filtered

In [ ]:
#rename P26_4 to P26_3 and P26_5 to P26_4 in orig.ident column of rna_adata_filtered
atac_adata_filtered.obs['orig.ident'] = atac_adata_filtered.obs['orig.ident'].str.replace('YM2_semitendinosus_2', 'YM2_2')
atac_adata_filtered.obs['orig.ident'] = atac_adata_filtered.obs['orig.ident'].str.replace('YM2_semitendinosus_3', 'YM3_3')
atac_adata_filtered.obs['orig.ident'].value_counts()

#### build mudata

In [ ]:
# ATAC layer must be raw counts — alias the long name
atac_adata_filtered.layers['counts'] = atac_adata_filtered.layers['Tn5_insertion_counts']

# Modality + donor on each modality's obs
rna_adata_filtered.obs['modality']  = 'expression'
atac_adata_filtered.obs['modality'] = 'accessibility'

In [ ]:
# Build mudata
import mudata as md, pandas as pd
mdata = md.MuData({'rna': rna_adata_filtered, 'atac': atac_adata_filtered})

# Promote to global
mdata.obs['modality'] = pd.concat([rna_adata_filtered.obs['modality'],
                                    atac_adata_filtered.obs['modality']])
mdata.obs['donor']    = pd.concat([rna_adata_filtered.obs['orig.ident'],
                                    atac_adata_filtered.obs['orig.ident']])
                                    # add sample column to obs
mdata.obs['sample'] = pd.concat([rna_adata_filtered.obs['sample'],
                                    atac_adata_filtered.obs['sample']])
mdata.update()

print(mdata)  # 21017 × (n_hvg + 78816), 2 modalities

In [ ]:
#write mudata to h5mu
mdata.write_h5mu('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/female_type2.h5mu')

## Padded mudata to fill in for both modalities in all cells

In [5]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import mudata as md

In [6]:
mdata = md.read_h5mu('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/female_type2.h5mu')
mdata

/projects/bgdb/asachan/.conda/envs/multivi/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bgdb/asachan/.conda/envs/multivi/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 21017 × 82816
  obs:	'modality', 'donor', 'sample'
  2 modalities
    rna:	13186 × 4000
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'modality'
      var:	'features', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'hvg', 'rank_genes_groups'
      obsm:	'X_umap'
      layers:	'counts', 'counts_float', 'lognorm'
    atac:	7831 × 78816
      obs:	'replicates', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'orig.ident', 'sample', 'group', 'ReadsInPeaks', 'FRIP', 'fiber_class_1_anno', 'Annotation', 'UMAP_1', 'UMAP_2', 'fiber_class_anno', 'country', 'age', 'Sex', 'modality'
      var:	'n_cells'
      obsm:	'X_UMAP'
      layers:	'Tn5_insertion_counts', 'counts'

In [7]:
# Re-load (assuming you have rna_adata and atac_peaks_adata still in scope or reload)
rna_adata = mdata['rna'].copy()
atac_peaks_adata = mdata['atac'].copy()

# Mark cells with their original modality membership BEFORE padding
rna_adata.obs['modality'] = 'expression'
atac_peaks_adata.obs['modality'] = 'accessibility'

# --- Pad RNA: add zero rows for ATAC-only cells ---
atac_only = atac_peaks_adata.obs_names.difference(rna_adata.obs_names)
rna_pad = ad.AnnData(
    X      = sp.csr_matrix((len(atac_only), rna_adata.n_vars), dtype=np.float32),
    obs    = pd.DataFrame(index=atac_only),
    var    = rna_adata.var.copy(),
    layers = {'counts': sp.csr_matrix((len(atac_only), rna_adata.n_vars), dtype=np.int32)},
)
rna_pad.obs['modality']   = 'accessibility'        # these are ATAC-only cells
rna_pad.obs['orig.ident'] = atac_peaks_adata.obs.loc[atac_only, 'orig.ident'].values
rna_pad.obs['sample']     = atac_peaks_adata.obs.loc[atac_only, 'sample'].values
rna_full = ad.concat([rna_adata, rna_pad], join='outer', merge='same')

# --- Pad ATAC: add zero rows for RNA-only cells ---
rna_only = rna_adata.obs_names.difference(atac_peaks_adata.obs_names)
atac_pad = ad.AnnData(
    X      = sp.csr_matrix((len(rna_only), atac_peaks_adata.n_vars), dtype=np.float32),
    obs    = pd.DataFrame(index=rna_only),
    var    = atac_peaks_adata.var.copy(),
    layers = {'counts': sp.csr_matrix((len(rna_only), atac_peaks_adata.n_vars), dtype=np.int32)},
)
atac_pad.obs['modality']   = 'expression'         # these are RNA-only cells
atac_pad.obs['orig.ident'] = rna_adata.obs.loc[rna_only, 'orig.ident'].values
atac_pad.obs['sample']     = rna_adata.obs.loc[rna_only, 'sample'].values
atac_full = ad.concat([atac_peaks_adata, atac_pad], join='outer', merge='same')

# --- Reorder to a single common index (critical for paired check) ---
all_cells = pd.Index(np.concatenate([rna_adata.obs_names, atac_only]))
# That's: all RNA cells first, then ATAC-only cells. Both modalities must use this order.
rna_full  = rna_full[all_cells].copy()
atac_full = atac_full[all_cells].copy()
assert (rna_full.obs_names == atac_full.obs_names).all()

# --- Build MuData ---
mdata = md.MuData({'rna': rna_full, 'atac': atac_full})
mdata.obs['modality']   = rna_full.obs['modality']     # both modalities have it identically
mdata.obs['orig.ident'] = rna_full.obs['orig.ident']
mdata.obs['sample']     = rna_full.obs['sample']
mdata.update()

print(mdata)
print(mdata.obs['modality'].value_counts())
# Expect: expression: 13186, accessibility: 7831, total: 21017

MuData object with n_obs × n_vars = 21017 × 82816
  obs:	'modality', 'orig.ident', 'sample'
  2 modalities
    rna:	21017 × 4000
      obs:	'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'modality'
      var:	'features', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      obsm:	'X_umap'
      layers:	'counts', 'counts_float', 'lognorm'
    atac:	21017 × 78816
      obs:	'replicates', 'TSSEnrichment', 'ReadsInTSS', 'ReadsInPromoter', 'ReadsInBlacklist', 'PromoterRatio', 'PassQC', 'NucleosomeRatio', 'nMultiFrags', 'nMonoFrags', 'nFrags', 'nDiFrags', 'BlacklistRatio', 'orig.ident', 'sample', 'group', 'ReadsInPeaks', 'FRIP', 'fiber_class_1_anno', 'Annotation', 'UMAP_1', 'UMAP_2', 'fiber_class_anno', 'country', 'age', 'Sex', 'modality'
      var:	'n_cells'
      obsm:	'X_UMAP'
      layers:	'Tn5_insertion_coun

/projects/bgdb/asachan/.conda/envs/multivi/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bgdb/asachan/.conda/envs/multivi/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/projects/bgdb/asachan/.conda/envs/multivi/lib/python3.11/site-packages/mudata/_core/mudata.

In [8]:
# 1. Index alignment — must be exact
assert (mdata['rna'].obs_names == mdata['atac'].obs_names).all()

# 2. Padding is actually zeros (not accidentally retaining values)
import numpy as np, scipy.sparse as sp

# expression-labeled cells: RNA real, ATAC should be zero
expr_mask = (mdata.obs['modality'] == 'expression').values
atac_X = mdata['atac'].layers['counts']
expr_atac_sum = np.asarray(atac_X[expr_mask].sum(axis=1)).ravel()
print(f"ATAC counts in RNA-only cells: max={expr_atac_sum.max()} (should be 0)")

# accessibility-labeled cells: ATAC real, RNA should be zero
acc_mask = (mdata.obs['modality'] == 'accessibility').values
rna_X = mdata['rna'].layers['counts']
acc_rna_sum = np.asarray(rna_X[acc_mask].sum(axis=1)).ravel()
print(f"RNA counts in ATAC-only cells: max={acc_rna_sum.max()} (should be 0)")

ATAC counts in RNA-only cells: max=0 (should be 0)
RNA counts in ATAC-only cells: max=0 (should be 0)


In [9]:
mdata.write_h5mu('/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/female_type2_padded.h5mu')

/projects/bgdb/asachan/.conda/envs/multivi/lib/python3.11/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/projects/bgdb/asachan/.conda/envs/multivi/lib/python3.11/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
